In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.simplefilter("ignore")

In [2]:
df_train = pd.read_csv("/kaggle/input/playground-series-s5e5/train.csv")
df_test = pd.read_csv("/kaggle/input/playground-series-s5e5/test.csv")

# Data Preprocessing

In [3]:
num_vars = df_train.drop(columns=['id','Calories']).select_dtypes(include=['int64', 'float64']).columns
cat_vars = ['Sex']

# Model

## XGB Baseline

In [4]:
SEED = 7

In [5]:
import xgboost as xgb
from sklearn.metrics import mean_squared_log_error
from sklearn.model_selection import train_test_split

In [6]:
X = df_train.drop(columns=['id', 'Calories'])
X = pd.get_dummies(X, columns=cat_vars, drop_first=True)
y = df_train['Calories']

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=SEED)

In [7]:
xgb_baseline = xgb.XGBRegressor(enable_calegorical=True)
xgb_baseline.fit(X_train, y_train)

y_val_pred = xgb_baseline.predict(X_val)
y_val_pred = np.maximum(y_val_pred, 0)
score = mean_squared_log_error(y_val_pred, y_val)
print(f'XGB Baseline Score: {score}')

XGB Baseline Score: 0.004272607917848008


## Big Tuna

In [8]:
import optuna
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold

In [9]:
def ensure_positive(y_pred):
    return np.maximum(y_pred, 0)  # Replace negative values with 0

def msle_eval(y_pred, dtrain):
    y_true = dtrain.get_label()
    y_pred = np.maximum(y_pred, 0)
    loss = mean_squared_log_error(y_true, y_pred)
    return 'MSLE', loss

# Function to run k-fold cross-validation with XGBoost and MSLE
def xgb_cv_msle(X, y, params, num_folds=5, num_boost_round=1000, early_stopping_rounds=50):
    kf = KFold(n_splits=num_folds, shuffle=True, random_state=SEED)
    fold_scores = []
    
    for train_idx, val_idx in kf.split(X):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        dtrain = xgb.DMatrix(X_train, label=y_train)
        dval = xgb.DMatrix(X_val, label=y_val)
        
        model = xgb.train(
            params,
            dtrain,
            num_boost_round=num_boost_round,
            evals=[(dval, 'val')],
            feval=msle_eval,
            early_stopping_rounds=early_stopping_rounds,
            verbose_eval=False
        )
        
        # Get predictions and ensure they're positive
        y_val_pred = model.predict(dval)
        y_val_pred = np.maximum(0, y_val_pred)
        score = mean_squared_log_error(y_val, y_val_pred)
        # print(score)
        fold_scores.append(score)
        
    return fold_scores

In [10]:
def objective(trial):
    params = {
        # "objective": "reg:squarederror",
        # "eval_metric": "rmse",
        "tree_method": "gpu_hist",
        "predictor": "gpu_predictor",
        "learning_rate": trial.suggest_float("learning_rate", 0.03, 0.1, step=0.01),
        "max_depth": trial.suggest_int("max_depth", 5, 20),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1),
        "max_bin": trial.suggest_int("max_bin", 256, 2048),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
        "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
        "grow_policy": trial.suggest_categorical("grow_policy", ["depthwise", "lossguide"]),
        "random_state": SEED
    }
    
    score = np.mean(xgb_cv_msle(X=X, y=y, params=params))
    return score

In [11]:
# %%time
# study = optuna.create_study(direction='minimize',
#                             sampler = optuna.samplers.RandomSampler(seed=SEED),
#                             study_name = "BIG BLUE FIN TUNA!!")
# study.optimize(objective, n_trials=100, show_progress_bar=True, )

In [12]:
# best_params = study.best_params
# print(f'Best Trial Params: {best_params}')

# print(f'Best Trial Value: {study.best_trial.value}')

In [13]:
# optuna.visualization.plot_optimization_history(study)

In [14]:
# optuna.visualization.plot_slice(study)

In [15]:
# optuna.visualization.plot_param_importances(study)

# Submission

In [16]:
best_model = xgb.XGBRegressor(
        device="cuda" if xgb.XGBRegressor().get_params().get("device") == "cuda" else "cpu",
        max_depth=10,
        colsample_bytree=0.7,
        subsample=0.9,
        n_estimators=2000,
        learning_rate=0.02,
        gamma=0.01, 
        max_delta_step=2,
        early_stopping_rounds=100,
        eval_metric="rmse",
        enable_categorical=True
    )

best_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100
)

[0]	validation_0-rmse:62.41628
[100]	validation_0-rmse:59.18832
[200]	validation_0-rmse:56.01445
[300]	validation_0-rmse:52.91826
[400]	validation_0-rmse:49.98219
[500]	validation_0-rmse:47.09408
[600]	validation_0-rmse:44.20208
[700]	validation_0-rmse:41.33031
[800]	validation_0-rmse:38.56041
[900]	validation_0-rmse:35.88776
[1000]	validation_0-rmse:33.32127
[1100]	validation_0-rmse:30.85038
[1200]	validation_0-rmse:28.57002
[1300]	validation_0-rmse:26.35975
[1400]	validation_0-rmse:24.22973
[1500]	validation_0-rmse:22.24732
[1600]	validation_0-rmse:20.42983
[1700]	validation_0-rmse:18.76012
[1800]	validation_0-rmse:17.21397
[1900]	validation_0-rmse:15.82557
[1999]	validation_0-rmse:14.58442


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.7, device='cpu', early_stopping_rounds=100,
             enable_categorical=True, eval_metric='rmse', feature_types=None,
             gamma=0.01, grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.02, max_bin=None,
             max_cat_threshold=None, max_cat_to_onehot=None, max_delta_step=2,
             max_depth=10, max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=2000,
             n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [17]:
# best_model = xgb.XGBRegressor(**best_params)
# best_model.fit(X, y)

X_test = df_test.drop(columns=['id'])
X_test = pd.get_dummies(X_test, columns=['Sex'], drop_first=True)
y_test_pred = best_model.predict(X_test)
y_test_pred = np.maximum(y_test_pred, 0)

submission = pd.read_csv("/kaggle/input/playground-series-s5e5/sample_submission.csv")
submission['Calories'] = y_test_pred
submission.to_csv('submission.csv', index=False)
submission.head()

,id,Calories
0,750000,28.513430
1,750001,107.332382
2,750002,86.070763
3,750003,133.646255
4,750004,76.397545


#